# 05 ANOVA

一元配置分散分析で4群の平均重心座標に差があるか確認するNotebook。

## 入力と出力

- 入力: `data/processed/{DATASET_NAME}/team_centroids_by_sequence.csv`
- 前提検定: `outputs/tables/{DATASET_NAME}/assumption_checks.csv`
- 出力: `outputs/tables/{DATASET_NAME}/anova.csv`
- 対象変数: `mean_centroid_x`, `mean_centroid_y`
- 比較群: `A_attack`, `A_defense`, `B_attack`, `B_defense`
- 有意水準: `alpha = 0.05`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import f_oneway

def find_project_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'はるた').exists():
            return candidate
    raise FileNotFoundError('プロジェクトルートが見つかりません。data/ と はるた/ がある場所で実行してください。')


PROJECT_ROOT = find_project_root()
HARUTA_ROOT = PROJECT_ROOT / 'はるた'
MATCH_DATE = '2023-11-25'
MATCH_ID = '117093'
MATCH_LABEL = 'tsukuba_vs_tsukuba-b'
DATASET_NAME = f'{MATCH_DATE}_{MATCH_ID}_{MATCH_LABEL}'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / DATASET_NAME
TABLES_DIR = HARUTA_ROOT / 'outputs' / 'tables' / DATASET_NAME
TABLES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / 'team_centroids_by_sequence.csv'
ASSUMPTION_PATH = TABLES_DIR / 'assumption_checks.csv'
OUTPUT_PATH = TABLES_DIR / 'anova.csv'

GROUP_COL = 'group'
GROUPS = ['A_attack', 'A_defense', 'B_attack', 'B_defense']
VALUE_COLS = ['mean_centroid_x', 'mean_centroid_y']
ALPHA = 0.05

## データ読み込み

In [ ]:
df = pd.read_csv(INPUT_PATH)
assumptions = pd.read_csv(ASSUMPTION_PATH)

df.head()

In [ ]:
df.groupby(GROUP_COL)[VALUE_COLS].agg(['count', 'mean', 'std'])

## 前提検定の確認

`mean_centroid_x` は一部グループでShapiro-Wilk検定が棄却されているため、ANOVAの解釈時に注意する。

In [ ]:
assumptions

## ANOVA実行

In [ ]:
def summarize_assumptions(assumptions_df, variable):
    shapiro_rows = assumptions_df[
        (assumptions_df['test'] == 'shapiro_wilk')
        & (assumptions_df['variable'] == variable)
    ]
    levene_rows = assumptions_df[
        (assumptions_df['test'] == 'levene')
        & (assumptions_df['variable'] == variable)
    ]
    shapiro_all_passed = bool(shapiro_rows['passed'].all())
    levene_passed = bool(levene_rows['passed'].all())
    failed_groups = shapiro_rows.loc[~shapiro_rows['passed'], 'group'].tolist()
    return shapiro_all_passed, levene_passed, failed_groups


def run_one_way_anova(data, variable, group_col=GROUP_COL, groups=GROUPS, alpha=ALPHA):
    samples = [data.loc[data[group_col] == group, variable].dropna().to_numpy() for group in groups]
    group_ns = {group: len(sample) for group, sample in zip(groups, samples)}

    statistic, p_value = f_oneway(*samples)
    all_values = np.concatenate(samples)
    grand_mean = all_values.mean()
    ss_between = sum(len(sample) * (sample.mean() - grand_mean) ** 2 for sample in samples)
    ss_within = sum(((sample - sample.mean()) ** 2).sum() for sample in samples)
    ss_total = ss_between + ss_within
    df_between = len(samples) - 1
    df_within = len(all_values) - len(samples)
    eta_squared = ss_between / ss_total if ss_total else np.nan

    shapiro_all_passed, levene_passed, failed_groups = summarize_assumptions(assumptions, variable)
    note = 'OK'
    if not shapiro_all_passed:
        note = f"正規性に注意: {', '.join(failed_groups)}"
    if not levene_passed:
        note = f"{note}; 等分散性に注意" if note != 'OK' else '等分散性に注意'

    return {
        'test': 'one_way_anova',
        'variable': variable,
        'groups': ','.join(groups),
        'n_total': len(all_values),
        'n_A_attack': group_ns.get('A_attack', 0),
        'n_A_defense': group_ns.get('A_defense', 0),
        'n_B_attack': group_ns.get('B_attack', 0),
        'n_B_defense': group_ns.get('B_defense', 0),
        'df_between': df_between,
        'df_within': df_within,
        'statistic': statistic,
        'p_value': p_value,
        'alpha': alpha,
        'significant': bool(p_value < alpha),
        'eta_squared': eta_squared,
        'shapiro_all_passed': shapiro_all_passed,
        'levene_passed': levene_passed,
        'note': note,
    }


anova_results = pd.DataFrame([run_one_way_anova(df, col) for col in VALUE_COLS])
anova_results

## 保存

In [ ]:
anova_results.to_csv(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')